In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/cardio_train.csv"
df = pd.read_csv(path)

print(df.shape)
df.head()

(70000, 13)


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [ ]:
if 'id' in df.columns:
    df = df.drop(columns=['id'])

df.head()

,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [ ]:
df['age_years'] = (df['age'] / 365.25).astype(int)
df = df.drop(columns=['age'])

df.head()

,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years
0,2,168,62.0,110,80,1,1,0,0,1,0,50
1,1,156,85.0,140,90,3,1,0,0,1,1,55
2,1,165,64.0,130,70,3,1,0,0,0,1,51
3,2,169,82.0,150,100,1,1,0,0,1,1,48
4,1,156,56.0,100,60,1,1,0,0,0,0,47


In [ ]:
X = df.drop(columns=['cardio'])
y = df['cardio']

print(X.shape)
print(y.shape)

(70000, 11)
(70000,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (49000, 11)
Validation: (10500, 11)
Test: (10500, 11)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

In [ ]:
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score,
    f1_score, roc_auc_score
)

def evaluate(y_true, y_pred, y_prob=None, label=""):

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob) if y_prob is not None else None

    print(f"\n--- {label} ---")
    print("Confusion Matrix [[TN FP],[FN TP]]:")
    print(cm)
    print(f"Accuracy: {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Sensitivity: {sensitivity:.3f}")
    print(f"Specificity: {specificity:.3f}")
    print(f"F1-score: {f1:.3f}")
    if auc is not None:
        print(f"AUC-ROC: {auc:.3f}")

In [ ]:
# Validation
rf_val_pred = rf.predict(X_val)
rf_val_prob = rf.predict_proba(X_val)[:, 1]

evaluate(y_val, rf_val_pred, rf_val_prob,
         label="Random Forest – Validation")

# Test
rf_test_pred = rf.predict(X_test)
rf_test_prob = rf.predict_proba(X_test)[:, 1]

evaluate(y_test, rf_test_pred, rf_test_prob,
         label="Random Forest – Test")


--- Random Forest – Validation ---
Confusion Matrix [[TN FP],[FN TP]]:
[[3748 1505]
 [1583 3664]]
Accuracy: 0.706
Precision: 0.709
Sensitivity: 0.698
Specificity: 0.713
F1-score: 0.704
AUC-ROC: 0.767

--- Random Forest – Test ---
Confusion Matrix [[TN FP],[FN TP]]:
[[3739 1514]
 [1548 3699]]
Accuracy: 0.708
Precision: 0.710
Sensitivity: 0.705
Specificity: 0.712
F1-score: 0.707
AUC-ROC: 0.763


In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=2000)
lr.fit(X_train, y_train)

lr_val_pred = lr.predict(X_val)
lr_val_prob = lr.predict_proba(X_val)[:, 1]

evaluate(y_val, lr_val_pred, lr_val_prob,
         label="Logistic Regression – Validation")


--- Logistic Regression – Validation ---
Confusion Matrix [[TN FP],[FN TP]]:
[[3960 1293]
 [1714 3533]]
Accuracy: 0.714
Precision: 0.732
Sensitivity: 0.673
Specificity: 0.754
F1-score: 0.701
AUC-ROC: 0.779


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    random_state=42,
    max_depth=6
)

dt.fit(X_train, y_train)

dt_val_pred = dt.predict(X_val)
dt_val_prob = dt.predict_proba(X_val)[:, 1]

evaluate(y_val, dt_val_pred, dt_val_prob,
         label="Decision Tree – Validation")


--- Decision Tree – Validation ---
Confusion Matrix [[TN FP],[FN TP]]:
[[4112 1141]
 [1714 3533]]
Accuracy: 0.728
Precision: 0.756
Sensitivity: 0.673
Specificity: 0.783
F1-score: 0.712
AUC-ROC: 0.791


Random Forest performed best overall with higher AUC-ROC and F1-score. It captured non-linear relationships better than Logistic Regression and Decision Tree. Logistic Regression was interpretable but slightly lower in sensitivity. Decision Tree showed moderate performance and possible overfitting. Since this task is disease screening, sensitivity is important, making Random Forest the most suitable model. A limitation observed was moderate specificity leading to some false positives.